# Building a Labelled Speech Corpus: Steps 1 to 4
**Course Project:** Speech Processing (Building a Labelled Speech Corpus from Online Videos)  
**Supervisor:** Prof. Mohsen Rashwan  
**Author:** Amr Gamal (`CaptainLevi04`)

---

### Pipeline Steps:
1. **Step 1 — Video Ingestion:** Download online video and parse metadata (yt-dlp).
2. **Step 2 — Audio Extraction:** Convert audio track to 16 kHz, mono, 16-bit PCM WAV (ffmpeg).
3. **Step 3 — Multi-Class Labelling:** Classify audio regions into `speech`, `music`, `mixed`, and `silence` with confidence scores (0 to 1), generate spectrogram, and support manual boundary adjustments.
4. **Step 4 — Pause-Aware Segmentation:** Cut speech and mixed regions into 5–15s chunks at natural pauses (VAD / energy pause detector) with 0.15s silence padding. Export segments index CSV and WAV files.

## 1. Setup & Environment

In [ ]:
# Install dependencies if running in fresh environment or Colab:
# !pip install -q yt-dlp numpy scipy pandas matplotlib soundfile torch

import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Audio, display

if "." not in sys.path:
    sys.path.append(".")

import speech_corpus as sc
print("Loaded speech_corpus version:", sc.__version__)

## 2. Configuration & Execution Mode
- `MODE = "DEMO"`: Instant execution on synthetic multi-class signal with ground truth (zero downloads, finishes in 5s).
- `MODE = "REAL"`: Download and process a web video (YouTube) or local file.

In [ ]:
MODE = "DEMO"  # "DEMO" or "REAL"

# If MODE == "REAL", set the video URL or local file path:
VIDEO_URL = "https://www.youtube.com/watch?v=jNQXAC9IVRw"
DATA_DIR = Path("data/demo_session" if MODE == "DEMO" else "data/real_session")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Mode: {MODE} | Output Directory: {DATA_DIR}")

## 3. Steps 1 & 2: Ingestion & Audio Extraction (16 kHz Mono WAV)

In [ ]:
if MODE == "DEMO":
    # Generate synthetic signal for immediate validation
    audio, gt_df = sc.build_mock_audio(seed=42, sr=sc.DEFAULT_SR)
    wav_path = DATA_DIR / "audio.wav"
    sc.write_wav(wav_path, audio, sample_rate=sc.DEFAULT_SR)
    metadata = {
        "title": "Synthetic Demonstration Audio",
        "channel": "Demo Channel",
        "duration_sec": len(audio) / sc.DEFAULT_SR,
        "license": "Creative Commons",
        "creative_commons": True
    }
else:
    video_path, meta = sc.download_video(VIDEO_URL, DATA_DIR, max_height=480)
    wav_path = sc.extract_audio(video_path, DATA_DIR / "audio.wav")
    sc.extract_audio_hq(video_path, DATA_DIR / "audio_hq.flac")
    audio, sr = sc.read_wav(wav_path)
    gt_df = None
    metadata = meta

print("Audio Information:")
print(sc.wav_info(wav_path))

# Audio preview:
display(Audio(audio[:sc.DEFAULT_SR * 30], rate=sc.DEFAULT_SR))

## 4. Step 3: Multi-Class Region Labelling

In [ ]:
CLASSIFIER_BACKEND = "heuristic"  # 'heuristic' or 'panns'

regions_df, probs = sc.classify_audio(audio, sr=sc.DEFAULT_SR, backend=CLASSIFIER_BACKEND)
regions_csv_path = DATA_DIR / "regions.csv"
regions_df.to_csv(regions_csv_path, index=False)

print(f"Total Regions Detected: {len(regions_df)}")
display(regions_df.head(10))

## 5. Step 3: Spectrogram Visualization (0–8 kHz)

In [ ]:
fig = sc.plot_spectrogram_regions(
    audio,
    regions_df,
    gt_df=gt_df if MODE == "DEMO" else None,
    sr=sc.DEFAULT_SR,
    title=f"Spectrogram & Region Annotations - {metadata['title']}",
    save_path=DATA_DIR / "regions_spectrogram.png"
)
plt.show()

## 6. Step 3: Evaluation Against Ground Truth

In [ ]:
if gt_df is not None:
    eval_res = sc.evaluate_ground_truth(regions_df, gt_df, total_dur=len(audio)/sc.DEFAULT_SR)
    print(f"Frame Accuracy:         {eval_res['frame_accuracy'] * 100:.2f}%")
    print(f"Median Boundary Error:  {eval_res['boundary_error_median_sec']:.2f} s")
    print(f"Max Boundary Error:     {eval_res['boundary_error_max_sec']:.2f} s")
    print("\nConfusion Matrix (% of truth):")
    display(eval_res['confusion_matrix_pct'])
else:
    print("Evaluation skipped (unlabelled real video).")

## 7. Manual Boundary Adjustment (Human Override)

In [ ]:
editor = sc.RegionEditor(regions_df)

# Example: adjust first boundary and split region 1
if len(regions_df) > 1:
    new_bound = regions_df.loc[0, 'end'] + 0.3
    editor.move_boundary(0, new_time=new_bound, by="Annotator")

if len(regions_df) > 1:
    midpoint = (regions_df.loc[1, 'start'] + regions_df.loc[1, 'end']) / 2.0
    editor.split_region(1, split_time=midpoint, by="Annotator")

edited_regions = editor.save(DATA_DIR / "regions_edited.csv", DATA_DIR / "regions_edits.csv")

print("Audit Log (regions_edits.csv):")
display(editor.get_edits_df())

## 8. Step 4: Segmentation into 5–15s Chunks at Real Pauses

In [ ]:
# Cut speech and mixed regions into 5-15s segments at real pauses
segments_df = sc.cut_speech_and_mixed_segments(
    audio=audio,
    regions_df=regions_df,
    out_dir=DATA_DIR,
    sr=sc.DEFAULT_SR,
    vad_backend="auto",
    pad_sec=0.15
)

print(f"Total Segments Generated: {len(segments_df)}")
display(segments_df[['segment_id', 'wav_file', 'start', 'end', 'duration', 'source_region_label', 'confidence', 'cut_type']].head(10))

## 9. Labeled Segments Spectrogram & Visual Report (First 20 Samples)

In [ ]:
# Display composite visual report: Zoomed-in Spectrogram + Visual Metadata Table
fig_segs = sc.plot_segments_overview(
    audio,
    segments_df,
    sr=sc.DEFAULT_SR,
    max_segments=20,
    save_path=DATA_DIR / "spectrogram.png"
)
plt.show()

## 10. Segment Duration Distribution (5–15s Verification)

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(segments_df['duration'], bins=10, color='steelblue', edgecolor='black', alpha=0.8)
plt.axvline(5.0, color='red', linestyle='--', linewidth=1.5, label='Min Limit (5s)')
plt.axvline(15.0, color='green', linestyle='--', linewidth=1.5, label='Max Limit (15s)')
plt.title("Distribution of Segment Durations (Target: 5-15s)", fontsize=11, fontweight='bold')
plt.xlabel("Duration (s)")
plt.ylabel("Count")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

print(f"Min Duration:  {segments_df['duration'].min():.2f} s")
print(f"Max Duration:  {segments_df['duration'].max():.2f} s")
print(f"Mean Duration: {segments_df['duration'].mean():.2f} s")

## 11. Summary & Handoff Tables

In [ ]:
summary_data = {
    "Audio Duration (min)": [round(len(audio) / sc.DEFAULT_SR / 60.0, 2)],
    "Total Classified Regions": [len(regions_df)],
    "Speech Regions": [int((regions_df['label'] == 'speech').sum())],
    "Music Regions": [int((regions_df['label'] == 'music').sum())],
    "Mixed Regions": [int((regions_df['label'] == 'mixed').sum())],
    "Silence Regions": [int((regions_df['label'] == 'silence').sum())],
    "Output 5-15s Segments": [len(segments_df)],
}

summary_df = pd.DataFrame(summary_data)
display(summary_df)
print(f"Output files saved to: {DATA_DIR.resolve()}")